# 00 — Pre-flight Checks

Verify infrastructure, credentials, and Feast config before running the pipeline.


In [1]:
%pip install -q kubernetes pyyaml redis pymilvus boto3

from _config import *
from pathlib import Path
from kubernetes import client

validate()

v1 = client.CoreV1Api()


Note: you may need to restart the kernel to use updated packages.
Namespace:  smartshop
S3:         http://minio.smartshop.svc.cluster.local:9000
Redis:      redis.smartshop.svc.cluster.local:6379
Milvus:     milvus.smartshop.svc.cluster.local:19530
Feast:      feast-smartshop-feast-registry.smartshop.svc.cluster.local:443
Config OK ✓


## Infrastructure connectivity


In [2]:
import socket

def check_tcp(host, port, timeout=5):
    try:
        s = socket.create_connection((host, port), timeout)
        s.close()
        return True
    except Exception:
        return False

INFRA = {
    "MinIO (S3)": (S3_ENDPOINT.replace("http://", "").split(":")[0],
                   int(S3_ENDPOINT.rsplit(":", 1)[1])),
    "Redis": (REDIS_HOST, REDIS_PORT),
    "Milvus": (MILVUS_HOST, MILVUS_PORT),
    "Feast Registry": (FEAST_REGISTRY.split(":")[0], int(FEAST_REGISTRY.split(":")[1])),
}

feast_cfg_exists = Path(FEAST_CLIENT_CONFIG).exists()
if feast_cfg_exists:
    print(f"Feast client config: {FEAST_CLIENT_CONFIG} \u2713")
    print(f"  (auto-mounted from FeatureStore CR)")
else:
    print(f"Feast client config: {FEAST_CLIENT_CONFIG} \u2717")
    print(f"  Attach the FeatureStore to this workbench in RHOAI dashboard:")
    print(f"  Data Science Projects \u2192 {NAMESPACE} \u2192 Workbenches \u2192 Edit \u2192 Connections \u2192 Feast")
print()

print("Infrastructure connectivity:")
print("-" * 60)
for name, (host, port) in INFRA.items():
    reachable = check_tcp(host, port)
    status = "\u2713" if reachable else "\u2717 unreachable"
    print(f"  {name:35s} {host}:{port}  {status}")


Feast client config: /opt/app-root/src/feast-config/smartshop ✓
  (auto-mounted from FeatureStore CR)

Infrastructure connectivity:
------------------------------------------------------------
  MinIO (S3)                          minio.smartshop.svc.cluster.local:9000  ✓
  Redis                               redis.smartshop.svc.cluster.local:6379  ✓
  Milvus                              milvus.smartshop.svc.cluster.local:19530  ✓
  Feast Registry                      feast-smartshop-feast-registry.smartshop.svc.cluster.local:443  ✓


## Secrets


In [3]:
REQUIRED_SECRETS = [
    S3_CREDENTIALS_SECRET,
    "feast-s3-credentials",
    "feast-redis-secret",
    HF_SECRET,
]

print("Secrets:")
print("-" * 60)
for sname in REQUIRED_SECRETS:
    try:
        v1.read_namespaced_secret(sname, NAMESPACE)
        print(f"  \u2713 {sname}")
    except client.ApiException:
        print(f"  \u2717 {sname}: MISSING")


Secrets:
------------------------------------------------------------
  ✓ smartshop-credentials
  ✓ feast-s3-credentials
  ✓ feast-redis-secret
  ✓ hf-credentials


## Readiness


In [4]:
def check_pod_ready(label_selector, ns=NAMESPACE):
    pods = v1.list_namespaced_pod(ns, label_selector=label_selector).items
    if not pods:
        return None, "no pods found"
    pod = pods[0]
    conditions = pod.status.conditions or []
    ready = any(c.type == "Ready" and c.status == "True" for c in conditions)
    return pod.metadata.name, "Ready" if ready else pod.status.phase

COMPONENTS = {
    "Feast": "feast.dev/name=smartshop-feast",
    "MinIO": "app=minio",
    "Redis": "app=redis",
}

print(f"{'=' * 60}")
print("READINESS CHECKLIST")
print(f"{'=' * 60}")

for label, selector in COMPONENTS.items():
    name, status = check_pod_ready(selector)
    icon = "\u2713" if status == "Ready" else "\u2717"
    print(f"  {icon} {label:25s} {name or 'N/A':45s} {status}")

try:
    v1.read_namespaced_config_map("feast-spark-engine", NAMESPACE)
    print(f"  \u2713 ConfigMap/feast-spark-engine")
except client.ApiException:
    print(f"  \u2717 ConfigMap/feast-spark-engine: MISSING")

print(f"{'=' * 60}")
print("\nPre-flight complete. Proceed to 01_data_pipeline.ipynb \u2192")


READINESS CHECKLIST
  ✓ Feast                     feast-smartshop-feast-6857b8d87d-9qd4c        Ready
  ✓ MinIO                     minio-5865759c84-cgsqc                        Ready
  ✓ Redis                     redis-5d666487c8-8qdvs                        Ready
  ✓ ConfigMap/feast-spark-engine

Pre-flight complete. Proceed to 01_data_pipeline.ipynb →
